In [1]:
! pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface llama-index-tools-google -U -q


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import login

login()

d:\HF AI Agent Course\hf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Create a FunctionTool

In [3]:
from llama_index.core.tools import FunctionTool

def get_weather(location: str)-> str:
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"

tool= FunctionTool.from_defaults(
    get_weather,
    name= "my_weather_tool",
    description= "Useful for getting weather of a location"
)

tool.call("Kathmandu")

Getting weather for Kathmandu


ToolOutput(blocks=[TextBlock(block_type='text', text='The weather in Kathmandu is sunny')], tool_name='my_weather_tool', raw_input={'args': ('Kathmandu',), 'kwargs': {}}, raw_output='The weather in Kathmandu is sunny', is_error=False)

Create QueryEngineTool

In [5]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

db= chromadb.PersistentClient(path= "./alfred_chroma_db")
chroma_collection= db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
llm = HuggingFaceInferenceAPI(model_name="meta-llama/Llama-3.2-3B-Instruct")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="some useful name",
    description="some useful description",
)
await tool.acall(
    "Responds about research on the impact of AI on the future of work and society?"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4421.93it/s]


ToolOutput(blocks=[TextBlock(block_type='text', text='Empty Response')], tool_name='some useful name', raw_input={'input': 'Responds about research on the impact of AI on the future of work and society?'}, raw_output=Response(response='Empty Response', source_nodes=[NodeWithScore(node=TextNode(id_='aa59dde1-d1be-4206-b3b2-fd47c2f62912', embedding=None, metadata={'file_path': 'd:\\HF AI Agent Course\\Unit 2\\data\\persona_1002.txt', 'file_name': 'persona_1002.txt', 'file_type': 'text/plain', 'file_size': 122, 'creation_date': '2026-09-16', 'last_modified_date': '2026-09-17'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='461b8ac6-fe0d-4805-8ba2-3ae963ab2997', node_type='4', metadata={'file_path': 'd:\\HF A

Create a ToolSpec

In [6]:
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()
tool_spec_list

In [7]:
[print(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

load_data load_data() -> List[llama_index.core.schema.Document]
Load emails from the user's account.
search_messages search_messages(query: str, max_results: Optional[int] = None)

        Searches email messages given a query string and the maximum number
        of results requested by the user
           Returns: List of relevant message objects up to the maximum number of results.

        Args:
            query (str): The user's query
            max_results (Optional[int]): The maximum number of search results
            to return.
create_draft create_draft(to: Optional[List[str]] = None, subject: Optional[str] = None, message: Optional[str] = None) -> str

        Create and insert a draft email.
           Print the returned draft's message and id.
           Returns: Draft object, including draft id and message meta data.

        Args:
            to (Optional[str]): The email addresses to send the message to
            subject (Optional[str]): The subject for the event
  

[None, None, None, None, None, None]